## 02 — Evaluation (base vs fine-tuned)

This notebook compares the **base** model vs **fine-tuned (LoRA)** on the held-out test split.

**Colab:** Runtime → Change runtime type → **GPU** (pick **T4** if listed) → Restart session → **Run all**.

- **Metrics:** BLEU, ROUGE-L, format compliance
- **Outputs:** `outputs/eval_metrics.json`, `outputs/qualitative_examples.md`

**New session?** The next cell can restore `lora_adapters` + `data/processed` from Google Drive if you saved them under `tatweer_challenge_submission`.


In [ ]:
# Colab setup (GPU required for Unsloth)
import os
os.environ["WANDB_DISABLED"] = "true"

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. In Colab: Runtime → Change runtime type → Hardware accelerator: GPU. "
        "Then Runtime → Restart session, and re-run this cell."
    )

!pip -q install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install -U "unsloth[colab]" transformers datasets accelerate peft trl bitsandbytes evaluate sacrebleu rouge-score scikit-learn matplotlib seaborn "pandas<3" "protobuf>=4.25.3,<6"

In [ ]:
# Repo bootstrap (prevents missing data/test.jsonl issues)
# If you opened this notebook from Drive, this cell ensures we run inside the cloned repo.

import os
from pathlib import Path

repo_dir = Path("/content/tatweer_challenge")
if not repo_dir.exists():
    !git clone -q https://github.com/Shamem-cyberx/tatweer_challenge.git /content/tatweer_challenge

os.chdir(str(repo_dir / "notebooks"))
print("cwd:", os.getcwd())
print("repo contents:", list(repo_dir.iterdir())[:5])

In [ ]:
# Optional: restore training artifacts from Google Drive (new Colab session wipes /content)
# Save path from notebook 01 copy: MyDrive/tatweer_challenge_submission
from pathlib import Path
import shutil

REPO = Path("/content/tatweer_challenge")
ADAPTER_MARKER = REPO / "outputs" / "lora_adapters" / "adapter_config.json"
TEST_JSONL = REPO / "data" / "processed" / "test.jsonl"

if not ADAPTER_MARKER.exists() or not TEST_JSONL.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        print("Not on Colab or Drive mount failed — ensure you ran notebook 01 in this session or copy files manually.")
    else:
        SRC = Path("/content/drive/MyDrive/tatweer_challenge_submission")
        if (SRC / "outputs" / "lora_adapters").exists():
            dst = REPO / "outputs" / "lora_adapters"
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(SRC / "outputs" / "lora_adapters", dst)
            print("Restored:", dst)
        if (SRC / "data" / "processed").exists():
            (REPO / "data" / "processed").mkdir(parents=True, exist_ok=True)
            for f in (SRC / "data" / "processed").glob("*.jsonl"):
                shutil.copy2(f, REPO / "data" / "processed" / f.name)
            print("Restored data/processed JSONL")
else:
    print("Artifacts already present — skip Drive restore.")

In [ ]:
# Load test set
from pathlib import Path
from datasets import load_dataset

root = Path("..").resolve()
data_dir = root / "data" / "processed"

ds = load_dataset(
    "json",
    data_files={"test": str(data_dir / "test.jsonl")},
)
test_ds = ds["test"]
len(test_ds), test_ds[0].keys()

In [ ]:
# Load base model and fine-tuned adapters
import torch
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_dir = (Path("..").resolve() / "outputs" / "lora_adapters")
if not (adapter_dir / "adapter_config.json").exists():
    raise FileNotFoundError(
        f"Missing LoRA adapters at {adapter_dir}. Run notebook 01 first or run the Drive restore cell above."
    )

base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

ft_model, _ = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

# Attach the trained LoRA adapters
from peft import PeftModel
ft_model = PeftModel.from_pretrained(ft_model, str(adapter_dir))

base_model.eval(); ft_model.eval();
FastLanguageModel.for_inference(base_model)
FastLanguageModel.for_inference(ft_model)

print("Adapter dir:", adapter_dir)
print("CUDA:", torch.cuda.is_available())

In [ ]:
# Inference helper

def generate(model, instruction: str, max_new_tokens: int = 256) -> str:
    prompt = "\n".join([
        "### Instruction:",
        instruction.strip(),
        "",
        "### Response:",
    ])
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    # Return only the completion part when possible
    return text.split("### Response:")[-1].strip()

# Quick sanity check
print("BASE:\n", generate(base_model, test_ds[0]["instruction"], 192)[:500])
print("\nFT:\n", generate(ft_model, test_ds[0]["instruction"], 192)[:500])

In [ ]:
# Quantitative evaluation
import sacrebleu
from rouge_score import rouge_scorer
import json

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def format_compliance(pred: str, expects_code: bool) -> int:
    if not expects_code:
        return 1
    # expects code: require a fenced code block and at least one PowerShell-ish token
    has_fence = "```" in pred
    has_cmd = any(tok in pred for tok in ["Get-", "Set-", "Test-", "Restart-", "Resolve-", "ipconfig", "sc.exe", "tracert"])
    return int(has_fence and has_cmd)


def run_eval(model):
    preds, refs = [], []
    fmt = []
    for ex in test_ds:
        p = generate(model, ex["instruction"], max_new_tokens=256)
        preds.append(p)
        refs.append(ex["response"].strip())
        fmt.append(format_compliance(p, bool(ex.get("expects_code", False))))

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    rougeL = sum(rouge.score(r, p)["rougeL"].fmeasure for r, p in zip(refs, preds)) / len(preds)
    fmt_acc = sum(fmt) / len(fmt)

    return {
        "bleu": float(bleu),
        "rougeL_f": float(rougeL),
        "format_compliance": float(fmt_acc),
        "preds": preds,
        "refs": refs,
    }

base_res = run_eval(base_model)
ft_res = run_eval(ft_model)

metrics = {
    "base": {k: v for k, v in base_res.items() if k in ["bleu", "rougeL_f", "format_compliance"]},
    "fine_tuned": {k: v for k, v in ft_res.items() if k in ["bleu", "rougeL_f", "format_compliance"]},
}
metrics

In [ ]:
# Save outputs + qualitative examples
from pathlib import Path

out_dir = Path("..").resolve() / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# 8 examples: mix good/bad by comparing rougeL per example
scores = []
for i, (ref, p_base, p_ft) in enumerate(zip(base_res["refs"], base_res["preds"], ft_res["preds"])):
    s_base = rouge.score(ref, p_base)["rougeL"].fmeasure
    s_ft = rouge.score(ref, p_ft)["rougeL"].fmeasure
    scores.append((i, s_base, s_ft))

# pick top improvements and worst regressions
scores_sorted = sorted(scores, key=lambda x: (x[2] - x[1]), reverse=True)
show_ids = [x[0] for x in scores_sorted[:4]] + [x[0] for x in scores_sorted[-4:]]

md_lines = ["## Qualitative examples (base vs fine-tuned)", ""]
for i in show_ids:
    ex = test_ds[i]
    md_lines += [
        f"### Example {i} — {ex.get('category','')}",
        "**Instruction**:",
        ex["instruction"],
        "",
        "**Reference**:",
        ex["response"],
        "",
        "**Base output**:",
        base_res["preds"][i],
        "",
        "**Fine-tuned output**:",
        ft_res["preds"][i],
        "",
        "---",
        "",
    ]

(out_dir / "qualitative_examples.md").write_text("\n".join(md_lines), encoding="utf-8")

print("Saved:", out_dir / "eval_metrics.json")
print("Saved:", out_dir / "qualitative_examples.md")
metrics